In [73]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
tf.random.set_seed(42)

In [74]:
dataset = pd.read_csv('/content/Reviews.csv')
df = dataset[['Summary','Score']]

In [75]:
#Tokenization
texts = df['Summary'].astype(str).tolist()
labels = df['Score'].values
labels = labels - 1
tokenizer = Tokenizer(num_words=10000,oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequence = tokenizer.texts_to_sequences(texts)

In [76]:
#padding
pad_sequence = pad_sequences(sequences= sequence,padding = 'post')
print(len(pad_sequence), len(labels))

568454 568454


In [77]:
#frequency of the each word
word_freq = tokenizer.word_counts

In [78]:
#splitting the dataset into train and test
x_train, x_test, y_train, y_test = train_test_split(pad_sequence, labels, test_size=0.2, random_state=42)


In [80]:
model = Sequential()
model.add(Embedding(input_dim=10000,output_dim=32))
model.add(LSTM(100,return_sequences=True))
model.add(LSTM(50))
model.add(Dense(5,activation='softmax'))
model.compile(loss = 'sparse_categorical_crossentropy',optimizer='Adam',metrics=['accuracy'])
print(model.summary())
model.fit(x_train,y_train,batch_size = 64,epochs = 10,validation_data=[x_test,y_test])

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 67s 9ms/step - accuracy: 0.6371 - loss: 1.1321 - val_accuracy: 0.6412 - val_loss: 1.1279
Epoch 2/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 81s 9ms/step - accuracy: 0.6618 - loss: 0.9826 - val_accuracy: 0.7215 - val_loss: 0.7494
Epoch 3/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 67s 9ms/step - accuracy: 0.7280 - loss: 0.7296 - val_accuracy: 0.7399 - val_loss: 0.7064
Epoch 4/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 68s 10ms/step - accuracy: 0.7473 - loss: 0.6817 - val_accuracy: 0.7478 - val_loss: 0.6915
Epoch 5/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 82s 10ms/step - accuracy: 0.7597 - loss: 0.6516 - val_accuracy: 0.7532 - val_loss: 0.6831
Epoch 6/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 65s 9ms/step - accuracy: 0.7695 - loss: 0.6260 - val_accuracy: 0.7568 - val_loss: 0.6795
Epoch 7/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 82s 9ms/step - accuracy: 0.7792 - loss: 0.6032 - val_accuracy: 0.7586 - val_loss: 0.6802
Epoch 8/10
7106/7106 ━━━━━━━━━━━━━━━━━━━━ 81s 9ms/step - accuracy: 0.7873 - 

In [81]:
scores = model.evaluate(x_test,y_test,verbose=0)
print(scores)
print("Accuracy: %.2f%%" % (scores[1]*100))

[0.6924774646759033, 0.7622678875923157]
Accuracy: 76.23%


In [91]:
#predection for new sentence
new_text = ["worst food"]
new_seq = tokenizer.texts_to_sequences(new_text)
maxlen = x_train.shape[1]
new_pad = pad_sequences(new_seq, maxlen=maxlen, padding='post')
prediction = model.predict(new_pad)
predicted_class = tf.argmax(prediction, axis=1).numpy()[0] + 1
print("Predicted score:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Predicted score: 1
